In [1]:
#importacion
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [2]:
# Rutas
ruta_carpeta = Path("../02_Tablas_de_resultados_evaluacion_ESP")
ruta_salida = Path("../03_Tabla_de_metricas_por_mes_por_cuenca")
ruta_salida.mkdir(parents=True, exist_ok=True)

In [3]:
# CRPS
def crps_fast(ens, obs):
    ens = np.sort(ens)
    n = len(ens)

    term1 = np.mean(np.abs(ens - obs))

    if n < 2:
        return np.nan

    diff = np.diff(ens)
    weights = np.arange(1, n) * np.arange(n - 1, 0, -1)
    term2 = np.sum(diff * weights) / (n**2)

    return term1 - term2

In [4]:
def crpss(df_lead, ens_cols):

    df = df_lead.copy()

    df["fecha_pronostico"] = pd.to_datetime(df["fecha_pronostico"])
    df["mes"] = df["fecha_pronostico"].dt.month
    df["year"] = df["fecha_pronostico"].dt.year

    # Precomputar climatología por mes y año (evita filtros caros)
    clim_dict = {}
    for mes in range(1, 13):
        df_mes = df[df["mes"] == mes]
        clim_dict[mes] = {
            y: df_mes[df_mes["year"] != y]["QC_hist"].dropna().values
            for y in df_mes["year"].unique()
        }

    pairs = []

    for row in df.itertuples(index=False):

        obs = row.QC_hist
        if pd.isna(obs):
            continue

        # ensamble
        ens = np.array([getattr(row, c) for c in ens_cols])
        ens = ens[~np.isnan(ens)]

        if len(ens) < 2:
            continue

        mes = row.mes
        year = row.year

        clim_ens = clim_dict.get(mes, {}).get(year, None)
        
        if clim_ens is None or len(clim_ens) < 2:
            continue

        crps_m = crps_fast(ens, obs)
        crps_r = crps_fast(clim_ens, obs)

        if np.isnan(crps_r) or crps_r == 0:
            continue

        pairs.append({
            "fecha": row.fecha_pronostico,
            "mes": mes,
            "crps_model": crps_m,
            "crps_ref": crps_r
        })

    return pd.DataFrame(pairs)

In [5]:
# Skill score - calcula el CRPSSad para una combinación de pares
def compute_crpss(df_pairs):

    if len(df_pairs) == 0:
        return np.nan

    crps_m = df_pairs["crps_model"].mean()
    crps_r = df_pairs["crps_ref"].mean()

    if crps_r == 0:
        return np.nan

    return 1 - (crps_m / crps_r)

In [6]:
# en df pairs tenemos todas las combinaciones de CRPS 
# Bootstrap por años

def bootstrap_crpss(df_pairs, n_boot=1000):

    df_pairs = df_pairs.copy()
    df_pairs["year"] = pd.to_datetime(df_pairs["fecha"]).dt.year #todos los años disponibles en df_pairs

    years = df_pairs["year"].unique()
    n_years = len(years) #numero de años disponibles

    results = []

    for _ in range(n_boot): #empieza una iteración para cada boot

        sampled_years = np.random.choice(years, size=n_years, replace=True) # se selecciona una combinación aleatoria de años 

        sample = pd.concat( #selecciona los pares de CRPS de cada año en sampled_years
            [df_pairs[df_pairs["year"] == y] for y in sampled_years],
            ignore_index=True
        )

        score = compute_crpss(sample) #calcula el CRPSS ad para la sample

        if not np.isnan(score):
            results.append(score)

    return np.array(results)


In [7]:
# =========================
# Loop principal
# =========================

archivos = list(ruta_carpeta.glob("*.csv"))

for ruta_csv in tqdm(archivos, desc="Procesando cuencas"):

    df = pd.read_csv(ruta_csv, low_memory=False)

    df["fecha_emision"] = pd.to_datetime(df["fecha_emision"])
    df["fecha_pronostico"] = pd.to_datetime(df["fecha_pronostico"])

    ens_cols = [c for c in df.columns if c.startswith("ens_") and c != "ens_prom"]

    resultados = []

    for lead, df_lead in df.groupby("lead"):

        df_pairs = crpss(df_lead, ens_cols)

        if df_pairs.empty:
            continue

        df_pairs["lead"] = lead
        df_pairs["year"] = df_pairs["fecha"].dt.year

        for mes, df_mes in df_pairs.groupby("mes"):

            if len(df_mes) < 10:
                continue

            # CRPSS determinístico
            crpss_val = compute_crpss(df_mes)

            # Bootstrap
            boot = bootstrap_crpss(df_mes, n_boot=300)

            if len(boot) == 0:
                continue

            p5, p25, p50, p75, p95 = np.percentile(boot, [5, 25, 50, 75, 95])

            resultados.append({
                "lead": lead,
                "mes": mes,
                "crpss": crpss_val,
                "crpss_med": p50,
                "p5": p5,
                "p25": p25,
                "p75": p75,
                "p95": p95,
                "n": len(df_mes)
            })

    if len(resultados) == 0:
        continue

    df_out = pd.DataFrame(resultados)

    df_out.to_csv(
        ruta_salida / f"{ruta_csv.stem}_crpss_bootstrap.csv",
        index=False
    )

Procesando cuencas: 100%|██████████| 47/47 [5:15:17<00:00, 402.50s/it]  
